In [ ]:
!pip install -q pypdf sentence-transformers chromadb

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

RUTA_PDF = "/content/apuntes.pdf"
TAM_FRAGMENTO = 250   #palabras por fragmento
SOLAPE = 50           #palabras compartidas entre fragmentos consecutivos
TOP_K = 4             #fragmentos a recuperar
MODELO = "intfloat/multilingual-e5-small"

In [ ]:
lector = PdfReader(RUTA_PDF)
palabras = []  # lista de (palabra, número de página)
for num_pagina, pagina in enumerate(lector.pages, start=1):
    texto = pagina.extract_text() or ""
    for palabra in texto.split():
        palabras.append((palabra, num_pagina))

print(f"Páginas: {len(lector.pages)} | Palabras extraídas: {len(palabras)}")
if len(palabras) == 0:
    print("Este PDF no tiene texto seleccionable (¿está escaneado?)")

In [ ]:
paso = TAM_FRAGMENTO - SOLAPE
fragmentos = []
for inicio in range(0, len(palabras), paso):
    bloque = palabras[inicio:inicio + TAM_FRAGMENTO]
    if len(bloque) < 50:
        break
    fragmentos.append({
        "texto": " ".join(p for p, _ in bloque),
        "pagina": bloque[0][1],
    })

print(f"Fragmentos creados: {len(fragmentos)}")
print(fragmentos[0]["texto"][:300], "...")

In [ ]:
modelo = SentenceTransformer(MODELO)
textos = ["passage: " + f["texto"] for f in fragmentos]
embeddings = modelo.encode(textos, normalize_embeddings=True, show_progress_bar=True)
print(embeddings.shape)

In [ ]:
cliente = chromadb.PersistentClient(path="/content/chroma_db")
try:
    cliente.delete_collection("apuntes")
except Exception:
    pass
coleccion = cliente.create_collection("apuntes")

coleccion.add(
    ids=[f"frag_{i}" for i in range(len(fragmentos))],
    documents=[f["texto"] for f in fragmentos],
    embeddings=embeddings.tolist(),
    metadatas=[{"pagina": f["pagina"]} for f in fragmentos],
)
print("Fragmentos indexados:", coleccion.count())

In [ ]:
def recuperar(tema, k=TOP_K):
    consulta = modelo.encode(["query: " + tema], normalize_embeddings=True)
    res = coleccion.query(query_embeddings=consulta.tolist(), n_results=k)
    for i in range(len(res["documents"][0])):
        pagina = res["metadatas"][0][i]["pagina"]
        distancia = res["distances"][0][i]
        print(f"--- Resultado {i+1} | página {pagina} | distancia {distancia:.3f}")
        print(res["documents"][0][i][:300], "...\n")

recuperar("escribe aquí un tema que sí esté en tu PDF")